# MCA-YOLO-A 复现 - Colab 训练

在 Google Colab 上用免费 GPU 训练 YOLOv8n baseline

## 使用说明
依次运行每个代码块（点击左侧 ▶ 按钮）即可。

In [ ]:
# 1. 检查 GPU
import torch
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 型号: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. 安装依赖
!pip install ultralytics -q
from ultralytics import YOLO
print("安装完成")

In [ ]:
# 3. 从 GitHub 拉取项目（包含代码和数据）
import os
REPO_PATH = '/content/mca-yolo-reproduce'

if not os.path.exists(REPO_PATH):
    !git clone https://github.com/Cranzz/mca-yolo-reproduce.git {REPO_PATH}
else:
    !cd {REPO_PATH} && git pull

%cd {REPO_PATH}
!ls data/

In [ ]:
# 4. 创建 YOLO 配置文件（指向本地数据）
yaml_content = f'''
path: {REPO_PATH}/data/yolo_format
train: train/images
val: val/images
test: test/images

nc: 4
names: ['D00', 'D10', 'D20', 'D40']
'''

with open(f'{REPO_PATH}/data/rdd2022.yaml', 'w') as f:
    f.write(yaml_content.strip())
print('配置文件已创建')

In [ ]:
# 5. 开始训练！（用 GPU）
model = YOLO('yolov8n.pt')

results = model.train(
    data=f'{REPO_PATH}/data/rdd2022.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    name='yolov8n_rdd2022_baseline_colab',
    device=0,
    plots=True,
)

print(f"训练完成！最佳模型保存在: {results.save_dir}")

In [ ]:
# 6. 下载结果到本地（可选）
# 训练完成后运行这个格子，浏览器会自动下载结果
from google.colab import files
import shutil

result_dir = '/content/runs/detect/yolov8n_rdd2022_baseline_colab'
if os.path.exists(result_dir):
    shutil.make_archive('/content/results', 'zip', result_dir)
    files.download('/content/results.zip')
    print('结果已下载到本地')
else:
    print('结果目录未找到')